# Step 7 — Training / evaluation debug (manifest + vertex head)

**Goal:** Run a minimal **train/val loop** on `DataRootDataset` rows: per-vertex VLM features, verb conditioning, optional SAM3D global latent, and **vertex affordance** labels from disk (no SAM3D inference in the loop).

**Prerequisites:**
- `torch`, project deps (`pip install -e ".[notebooks,dev]"`).
- **Example pack** under `examples/data_manifest/` (tracked mesh + labels). Optional **projection cache** files are **not** tracked; generate them once:
  ```bash
  python scripts/build_example_training_fixtures.py
  ```
- For your own data: add `vertex_semantics_path` (same layout as notebook **05** `vertex_semantic.pt`) and optional `sam3d_global_latent_path` to each manifest line — see `docs/data_layout.md`.

**Pass checklist (last cell):**
- Train + val datasets load without error
- Training loss decreases over a few epochs on the toy split
- Val BCE is finite and not NaN

In [ ]:
import subprocess
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
for ROOT in [_cwd, *_cwd.parents]:
    if (ROOT / "pyproject.toml").is_file() and (ROOT / "src").is_dir():
        if str(ROOT / "src") not in sys.path:
            sys.path.insert(0, str(ROOT / "src"))
        break
else:
    raise RuntimeError("Run from the repo or notebooks/.")

EXAMPLE = ROOT / "examples" / "data_manifest"
feat_pt = EXAMPLE / "features" / "tiny_vertex_semantic.pt"
if not feat_pt.is_file():
    subprocess.check_call([sys.executable, str(ROOT / "scripts" / "build_example_training_fixtures.py")], cwd=str(ROOT))
    assert feat_pt.is_file(), "Fixture script did not create tiny_vertex_semantic.pt"
print("Example fixtures OK:", feat_pt)

In [ ]:
from utils.config import load_config, project_root

ROOT = project_root()
cfg = load_config()

MANIFEST_ROOT = ROOT / "examples" / "data_manifest"
MANIFEST_PATH = MANIFEST_ROOT / "manifest_training.jsonl"

N_EPOCHS = 5
LR = float(cfg.get("training", {}).get("learning_rate", 1e-4))
DEVICE = __import__("torch").device("cuda" if __import__("torch").cuda.is_available() else "cpu")
print("ROOT", ROOT)
print("manifest", MANIFEST_PATH)
print("device", DEVICE)

In [ ]:
from datasets.data_root_dataset import DataRootDataset

ds_train = DataRootDataset(
    data_root=MANIFEST_ROOT,
    manifest_path=MANIFEST_PATH,
    cfg=cfg,
    split="train",
    load_mesh_eager=False,
    load_vertex_labels_eager=True,
)
ds_val = DataRootDataset(
    data_root=MANIFEST_ROOT,
    manifest_path=MANIFEST_PATH,
    cfg=cfg,
    split="val",
    load_mesh_eager=False,
    load_vertex_labels_eager=True,
)
print("train samples:", len(ds_train), "| val:", len(ds_val))
print("keys[0]:", sorted(ds_train[0].keys()))

In [ ]:
import torch
from vlm.text_encoder import encode_texts
from vlm.vlm_wrapper import VLMWrapper

verbs = sorted({ds_train[i]["verb"] for i in range(len(ds_train))} | {ds_val[i]["verb"] for i in range(len(ds_val))})
vlm = VLMWrapper()
emb_mat = encode_texts(vlm, verbs)
verb_embeddings = {v: emb_mat[i].clone() for i, v in enumerate(verbs)}
print("verbs:", verbs, "| dim", next(iter(verb_embeddings.values())).shape[0])

In [ ]:
from models.mlp_head import build_affordance_mlp

use_sam3d = ds_train[0].get("sam3d_global_latent") is not None
model = build_affordance_mlp(cfg, include_sam3d=use_sam3d).to(DEVICE)
if use_sam3d:
    z = ds_train[0]["sam3d_global_latent"]
    assert z.numel() == model.cfg.sam3d_dim, (z.shape, model.cfg.sam3d_dim)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
print(model)
print("use_sam3d:", use_sam3d)

In [ ]:
from training.vertex_affordance_train import eval_vertex_bce, training_epoch_vertex_bce

train_losses: list[float] = []
val_losses: list[float] = []
for ep in range(N_EPOCHS):
    tr = training_epoch_vertex_bce(model, optimizer, ds_train, verb_embeddings=verb_embeddings, device=DEVICE)
    va = eval_vertex_bce(model, ds_val, verb_embeddings=verb_embeddings, device=DEVICE)
    train_losses.append(tr)
    val_losses.append(va)
    print(f"epoch {ep+1}/{N_EPOCHS}  train {tr:.4f}  val {va:.4f}")

import math

assert all(math.isfinite(x) for x in train_losses + val_losses)
assert train_losses[-1] <= train_losses[0] + 0.5, "train loss should not diverge on the toy split"


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(train_losses, label="train BCE")
ax.plot(val_losses, label="val BCE")
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.legend()
ax.set_title("Vertex affordance — toy manifest")
plt.tight_layout()
plt.show()

assert train_losses[0] >= train_losses[-1] * 0.5, "training loss should generally drop on this toy task"
print("PASS: Step 7 train/val loop")